## Regression model

### log log regression

In [4]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

def calculate_price_elasticity(historical_data):
    """
    historical_data contains: date, price, quantity_sold, competitors_price, seasonality
    Returns: price elasticity coefficient
    """
    # Log transformation for elasticity interpretation
    historical_data['log_quantity'] = np.log(historical_data['quantity_sold'])
    historical_data['log_price'] = np.log(historical_data['price'])
    historical_data['log_comp_price'] = np.log(historical_data['competitor_price'])
    
    # Add control variables
    X = historical_data[['log_price', 'log_comp_price', 'seasonality']]
    X = sm.add_constant(X)
    y = historical_data['log_quantity']
    
    model = sm.OLS(y, X).fit()
    
    # Coefficient of log_price is the price elasticity
    price_elasticity = model.params['log_price']
    
    return price_elasticity, model

### AB Testing

In [5]:
def calculate_elasticity_from_ab_test(control_price, test_price, 
                                       control_demand, test_demand):
    """
    Calculate elasticity from controlled experiments
    """
    pct_change_price = (test_price - control_price) / control_price
    pct_change_demand = (test_demand - control_demand) / control_demand
    
    elasticity = pct_change_demand / pct_change_price
    
    return elasticity

### Instrumental Variables (for endogeneity)

In [9]:
pip install linearmodels

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 1.5/1.5 MB 24.3 MB/s  0:00:00

   ------------------------------ --------- 3/4 [linearmodels]
   ------------------------------ --------- 3/4 [linearmodels]
   ---------------------------------------- 4/4 [linearmodels]

Note: you may need to restart the kernel to use updated packages.


In [11]:
from linearmodels.iv import IV2SLS

def calculate_elasticity_with_iv(data):
    """
    Use instrumental variables to handle price endogeneity
    (price may be correlated with unobserved demand shocks)
    """
    # Instrument: cost shocks, competitor prices, etc.
    formula = 'log_quantity ~ 1 + [log_price ~ cost_shocks] + log_comp_price + seasonality'
    model = IV2SLS.from_formula(formula, data)
    results = model.fit()
    
    return results.params['log_price']


In [12]:
### Double ML

In [13]:
pip install econml

   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ---------------------------------------- 2.3/2.3 MB 25.0 MB/s  0:00:00
   ---------------------------------------- 0.0/11.1 MB ? eta -:--:--
   -------------------------- ------------- 7.3/11.1 MB 36.0 MB/s eta 0:00:01
   ---------------------------------------- 11.1/11.1 MB 34.0 MB/s  0:00:00
   ---------------------------------------- 0.0/545.1 kB ? eta -:--:--
   ---------------------------------------- 545.1/545.1 kB 23.4 MB/s  0:00:00
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 1.5/1.5 MB 30.0 MB/s  0:00:00

   ------ --------------------------------- 1/6 [sparse]
  Attempting uninstall: scikit-learn
   ------ --------------------------------- 1/6 [sparse]
    Found existing installation: scikit-learn 1.7.2
   ------ --------------------------------- 1/6 [sparse]
   ------------- -------------------------- 2/6 [scikit-learn]
    Uninstalling 

In [14]:
from econml.dml import LinearDML
from sklearn.ensemble import RandomForestRegressor

def calculate_elasticity_with_ml(data):
    """
    Use machine learning to control for confounders
    """
    Y = data['quantity_sold']  # Outcome
    T = data['price']  # Treatment (price)
    X = data[['competitor_price', 'seasonality', 'marketing_spend']]  # Controls
    
    # Double Machine Learning
    est = LinearDML(model_y=RandomForestRegressor(),
                    model_t=RandomForestRegressor())
    est.fit(Y, T, X=X)
    
    elasticity = est.coef_[0]
    return elasticity